# Plan 6 — Colab Single-GPU Deep Learning & LLM Runner (L4 24GB / A100 40–80GB)

This notebook runs Section 3.3 of the unified CS482 Data Analysis pipeline (`scripts/06_dl_and_llm_gpu.py`): cloning or pulling `main`, verifying a Colab **NVIDIA L4 (24GB)** or **NVIDIA A100 (40GB / 80GB)** runtime, installing the GPU deep learning stack (`transformers>=5.10` with native `gemma4_unified` & `qwen3_5` support, `peft`, `bitsandbytes`, `accelerate`, `chronos-forecasting`), training and evaluating `PatchTST` (1M / 15M), `Chronos-2` / `Chronos-T5-Large`, `FinBERT` (110M), and 4-bit NF4 QLoRA LLM fusion against the **cryptographically frozen dataset** (`data/frozen/`, read-only), printing Table 2 (`outputs/tables/table2_h3_ablation.md`), and packaging the H3 GPU artifacts into `dl_gpu_outputs.zip`.

- **Automatic VRAM-aware LLM selection (`--llm-model auto`)**:
  - **Colab L4 (24GB VRAM, `< 35 GB`)**: automatically selects **`google/gemma-4-12B-it`** (`11.96B` ungated `Gemma4UnifiedForConditionalGeneration`, `~15.3 GB` peak VRAM, zero OOM; override to `google/gemma-4-12B` or `Qwen/Qwen3-14B` if desired).
  - **Colab A100 (40GB / 80GB VRAM, `>= 35 GB`)**: automatically selects **`Qwen/Qwen3.8-27B`** (`Qwen3_5ForConditionalGeneration`, ungated; optional `Qwen/Qwen3-32B` or `google/gemma-4-31B-it`).
- After downloading `dl_gpu_outputs.zip` into the repository, run `uv run python scripts/07_build_report.py` locally to recompile `report/CS482_GroupX_StockVolatilityLLM.pdf` and verify all 61 empirical claims.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Lc-wing/data_science_project.git"
BRANCH = "main"
PLAN_DIR = "plan_6_stock_volatility"
ENTRY = "scripts/06_dl_and_llm_gpu.py"

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)

cwd = Path.cwd().resolve()
if (cwd / ENTRY).exists() and (cwd.parent / ".git").exists():
    top = cwd.parent
elif (cwd / ".git").exists() and (cwd / PLAN_DIR).is_dir():
    top = cwd
else:
    top = (cwd / "data_science_project").resolve()

if (top / ".git").exists():
    for args in (("fetch", "origin", BRANCH), ("checkout", BRANCH), ("pull", "origin", BRANCH)):
        done = git(*args, cwd=top)
        assert done.returncode == 0, done.stderr
else:
    done = git("clone", "--branch", BRANCH, REPO, str(top))
    assert done.returncode == 0, done.stderr

plan_root = top / PLAN_DIR
assert (plan_root / ENTRY).exists(), f"{ENTRY} not found under {plan_root}"
os.chdir(plan_root)
print("Working directory:", Path.cwd())


In [ ]:
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to NVIDIA L4 (24GB) or A100 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
default_llm = "Qwen/Qwen3.8-27B" if vram_gb >= 35.0 else "google/gemma-4-12B-it"
print(f"Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) -> auto LLM: {default_llm}")

# Install GPU deep learning stack (transformers>=5.10 for gemma4_unified & qwen3_5 support)
!pip install -q -U "transformers>=5.10" peft bitsandbytes accelerate chronos-forecasting
!pip install -q -U pyarrow lightgbm arch shap
!pip install -q -e . --no-deps


In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Override to "google/gemma-4-12B" or "Qwen/Qwen3-14B" if desired; "auto" picks L4/A100 best
LLM_MODEL = "auto"

# Run H3 Deep Time-Series & LLM pipeline (--resume reuses already-saved Stage 1-4 outputs)
!python scripts/06_dl_and_llm_gpu.py --allow-weight-download --resume \
    --llm-model {LLM_MODEL} --llm-epochs 1


In [ ]:
import zipfile
from pathlib import Path

print(Path("outputs/tables/table2_h3_ablation.md").read_text())

export_patterns = (
    "outputs/predictions/patchtst_*.parquet",
    "outputs/predictions/chronos_*.parquet",
    "outputs/predictions/finbert_*.parquet",
    "outputs/predictions/llm_*.parquet",
    "outputs/predictions/lgbm_plus_*.parquet",
    "outputs/metrics/patchtst_*_metrics.json",
    "outputs/metrics/chronos_*_metrics.json",
    "outputs/metrics/llm_run_metadata.json",
    "outputs/metrics/h3_dl_summary.json",
    "outputs/tables/table2_h3_ablation.*",
)
archive_path = Path("dl_gpu_outputs.zip").resolve()
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in export_patterns:
        for fp in sorted(Path(".").glob(pat)):
            zf.write(fp, arcname=fp.as_posix())
print("Packaged archive for download ->", archive_path)
